# Bronze Layer — raw copy of Lakebase Postgres (Incremental, append-only)

Read **videos / channels / comments / search_results** + **places / place_search_results / search_runs** (search on demand, từ 2026-10-31) từ Lakebase Postgres → **append** vào `prod.bronze.*_raw`. Không transform: đúng cột nguồn + `ingest_ts` / `ingest_date`.

**Why:** Postgres giữ latest values only (UPSERT), nên Bronze là nơi duy nhất có thể replay để rebuild Silver nếu step sau có bug.

**Incremental Flow (per table):**
1. Lấy watermark (`MAX(COALESCE(updated_at, created_at))`, search_results: `MAX(fetched_at)`) từ Bronze table
2. SELECT từ `lakebase_pg.public.<table>`: chỉ rows có watermark column > watermark
3. APPEND vào `prod.bronze.<table>_raw` (partitioned by `ingest_date`)
4. Retention: DELETE partitions older than 180 days
5. Verify row counts

> **Note:** Đọc qua foreign catalog `lakebase_pg` thay vì JDBC để tránh timezone mismatch giữa Spark (UTC) và Postgres.


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SOURCE_CATALOG = "lakebase_pg.public"  # Lakebase Postgres foreign catalog
BRONZE_SCHEMA = BRONZE
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"  # business day used for ingest_date
RETENTION_DAYS = 180

# source table -> watermark expression
SOURCES = {
    "channels": "COALESCE(updated_at, created_at)",
    "videos": "COALESCE(updated_at, created_at)",
    "comments": "COALESCE(updated_at, created_at)",
    "search_results": "fetched_at",  # append-only log, never updated
    # on-demand search job (project1_search); tables exist after its first real run
    "places": "COALESCE(updated_at, created_at)",
    "place_search_results": "fetched_at",  # append-only
    "search_runs": "finished_at",  # one row per run, written once at the end
    "search_keywords": "COALESCE(updated_at, created_at)",  # auto_search schedule (upserted)
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
for name in SOURCES:
    print(f"{SOURCE_CATALOG}.{name} → {BRONZE_SCHEMA}.{name}_raw")


In [ ]:
# --- Ingest: one ingest_ts for the whole run, append new reads ---
from pyspark.sql.functions import lit, to_date, from_utc_timestamp

ingest_ts = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]
print(f"🕒 ingest_ts: {ingest_ts}")

loaded = {}
for name, wm_expr in SOURCES.items():
    source_table = f"{SOURCE_CATALOG}.{name}"
    bronze_table = f"{BRONZE_SCHEMA}.{name}_raw"
    if not spark.catalog.tableExists(source_table):
        print(f"⏭️ {name}: {source_table} does not exist yet, skipped")
        continue

    # Create the Bronze table on first run: source columns + ingest metadata, partitioned by ingest_date
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {bronze_table}
        USING DELTA
        PARTITIONED BY (ingest_date)
        AS SELECT *, CAST(NULL AS TIMESTAMP) AS ingest_ts, CAST(NULL AS DATE) AS ingest_date
        FROM {source_table} WHERE 1 = 0
    """)

    # Schema drift: add new source columns to Bronze (older reads keep NULL)
    bronze_cols = set(spark.table(bronze_table).columns)
    new_cols = [f for f in spark.table(source_table).schema.fields if f.name not in bronze_cols]
    if new_cols:
        spark.sql(f"ALTER TABLE {bronze_table} ADD COLUMNS (" + ", ".join(f"{f.name} {f.dataType.simpleString()}" for f in new_cols) + ")")
        print(f"➕ {name}: added columns {[f.name for f in new_cols]} to {bronze_table}")

    watermark = spark.sql(f"SELECT MAX({wm_expr}) AS wm FROM {bronze_table}").collect()[0]["wm"]
    if watermark:
        src_df = spark.sql(f"""
            SELECT * FROM {source_table}
            WHERE {wm_expr} > '{watermark.strftime("%Y-%m-%d %H:%M:%S.%f")}'
        """)
        print(f"💧 {name}: watermark {watermark}")
    else:
        src_df = spark.sql(f"SELECT * FROM {source_table}")
        print(f"📦 {name}: full load (no watermark — first run)")

    src_df = (
        src_df.withColumn("ingest_ts", lit(ingest_ts))
        .withColumn("ingest_date", to_date(from_utc_timestamp(lit(ingest_ts), SNAPSHOT_TZ)))
    )
    loaded[name] = src_df.count()
    if loaded[name] > 0:
        src_df.write.format("delta").mode("append").saveAsTable(bronze_table)
    print(f"✅ {name}: appended {loaded[name]} rows to {bronze_table}")


In [ ]:
# --- Retention: drop reads older than RETENTION_DAYS (Silver keeps the history) ---
for name in SOURCES:
    bronze_table = f"{BRONZE_SCHEMA}.{name}_raw"
    if not spark.catalog.tableExists(bronze_table):
        continue
    spark.sql(f"DELETE FROM {bronze_table} WHERE ingest_date < date_sub(current_date(), {RETENTION_DAYS})")
print(f"🧹 Retention applied: keep last {RETENTION_DAYS} days")


In [ ]:
# --- Verify ---
for name in SOURCES:
    bronze_table = f"{BRONZE_SCHEMA}.{name}_raw"
    if not spark.catalog.tableExists(bronze_table):
        continue
    stats = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT ingest_date) AS days, MAX(ingest_ts) AS last_ingest FROM {bronze_table}").collect()[0]
    print(f"📊 {bronze_table}: {stats['n']} rows over {stats['days']} day(s), last ingest {stats['last_ingest']}, loaded this run: {loaded.get(name, 0)}")
print("\n--- Bronze ingest complete ---")
